# Anime3D Colab — 通用 Image-to-3D 執行入口
依 AI_Agent_Colab_Anime3D_Pipeline 執行 N00–N08。角色可選用 character-colab/2 的獨立外觀評分閘門（先建模、評分後才綁骨）。預設只檢查與規劃，GPU 後端預設 Pixal3D；Tripo 另需明確付費授權。未核准立繪、缺模型環境、骨架／動畫未通過均不可宣稱完成。

要測試尚未發布的修復，先上傳交付的 `anime3d_colab_source.zip` 至 `/content/`；否則使用 GitHub 指定版本。研究模型請先依官方文件安裝到獨立 Python/CUDA 環境並填入 backend config；此 notebook 不會猜測或混裝模型依賴。

個別角色、參考圖及 asset_spec 只放 Google Drive。角色新版流程先生成 GPU 模型，於 qa/appearance_review.json 真實提交評分後才允許執行 post_gpu 綁骨；詳見 docs/pipeline/CHARACTER_COLAB_QUALITY.md。


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile
from google.colab import drive, userdata
drive.mount('/content/drive')
REPO = Path('/content/cosmos_chimera_forge')
SOURCE_ARCHIVE = Path('/content/anime3d_colab_source.zip')
REPO_REVISION = 'main'  # 可改成已發布且查核過的 commit SHA
if SOURCE_ARCHIVE.is_file():
    with zipfile.ZipFile(SOURCE_ARCHIVE) as bundle:
        for item in bundle.infolist():
            target = (REPO / item.filename).resolve()
            if not target.is_relative_to(REPO.resolve()):
                raise ValueError('Unsafe source archive path')
        bundle.extractall(REPO)
    print('Source: uploaded local verification bundle')
elif not REPO.is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/WaylenChang/cosmos_chimera_forge.git', str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', REPO_REVISION], check=True)
if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], check=True)
assert (REPO / 'scripts/anime3d_preflight.py').is_file(), 'Source revision lacks this notebook support; upload the verification bundle'


In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/project/cosmos_chimera_forge'
DRIVE_PROJECT = str(Path(DRIVE_ROOT) / 'characters')  # runner 的 --root 必須指向分類根目錄
SPEC = str(Path(DRIVE_PROJECT) / '<subject_id>' / '00_settings' / 'asset_spec.json')  # 在 Drive 選定真實個別設定
if not Path(SPEC).is_file():
    raise FileNotFoundError('Provide a real Drive character asset_spec.json; do not use a GitHub example')
ASSET_ID = json.loads(Path(SPEC).read_text(encoding='utf8'))['asset_id']
EXECUTE = False
STAGE = 'pre_gpu'  # 先 pre_gpu；再 gpu 初始模型；外觀審查後執行 post_gpu
BATCH_QUEUE = ''  # 選填 batch_queue.json 路徑；設定後一次批次跑完再關機
AUTO_SHUTDOWN_GPU = True  # STAGE=='gpu' 完成、佇列為空或閒置逾時時自動關閉 Colab GPU
GPU_IDLE_TIMEOUT_SEC = 180
BACKEND = 'pixal3d'  # pixal3d / anigen / partcrafter / tripo
RIG_BACKEND = 'skintokens'  # Pixal3D mesh → learned skeleton + vertex weights; 'none' for an already rigged input
PIPELINE = 'colab'
BACKEND_CONFIG = ''  # 例如 /content/drive/MyDrive/Anime3D_Agent/backends.json
GENERATOR_EXECUTABLE = ''  # 選填自訂 adapter，可執行檔路徑
EXECUTE_PAID_API = False
INSTALL_BLENDER = True  # 僅 STAGE in ('post_gpu', 'all') 且 EXECUTE=True 時安裝，不在 GPU 階段安裝
BLENDER = shutil.which('blender') or ''
if not Path(SPEC).is_file():
    raise FileNotFoundError('Select existing approved character asset_spec.json in Google Drive before proceeding')
spec = json.loads(Path(SPEC).read_text(encoding='utf8'))
assert spec['input_mode'] == 'image'
print('Subject:', spec.get('subject_name'), spec['subject_id'], 'Source:', spec['source_pack'])


## N00–N02：來源、視角與環境
來源可位於 `01_portraits/` 或 `03_completed/portraits/`；必須有真實 `reviews/<source_pack>.json` 且 status=ok。範例不會替立繪建立核准紀錄。Pixal3D mesh 後接獨立 SkinTokens runtime（官方 demo.py --use_transfer）；請在 BACKEND_CONFIG 同時提供 pixal3d 與 skintokens 設定。參考 docs/anime3d_backends.example.json。原始 mesh 留在 source/，真正 skin/rig 留在 rig/，均以 SHA256 續跑。

SkinTokens 請依 https://github.com/VAST-AI-Research/SkinTokens 在獨立 Python 3.11 環境安裝官方 Torch/CUDA、requirements.txt、flash-attn，再執行 download.py --model。Pixal3D 請依 https://github.com/TencentARC/Pixal3D 安裝 TRELLIS.2 基底、requirements.txt、natten 與 utils3d。兩個模型不可混裝到同一環境。GPU 最低需求依官方與實際 compute capability 檢查；不得因 Colab 分到較小 GPU 就宣稱可推論。

adapter 已提供並有結構／失敗／續跑測試，但尚未通過本角色的真實 GPU 測試。SkinTokens 不保證標準人形骨名；動作仍需依實際骨架映射與視覺驗收，不能把 envelope 骨架或不存在的動畫當正式成果。

新版角色 source spec 需 `character_colab_workflow: "character-colab/2"` 與 `reference_views`（front、turnaround 或 side/back、three_quarter_45）。合圖不能當成已校準多視角，`multiview_dir` 仍需真實相機 `transforms.json`。GPU 階段僅留下原始 source GLB，外觀評分不合格時先提出修復與選模建議，不自動反覆 GPU 推論。

注意：目前 Python runner 的已核准來源仍採分類根目錄下 `01_portraits/<source_pack>/` 或 `03_completed/portraits/<source_pack>/`，不會憑空把 `<subject_id>/` 視作已核准來源；三層實際 Drive 目錄需先使用經驗證的相容轉接或更新校驗器才可執行。不能因檔案存在就跳過人工核准。


In [ ]:
if EXECUTE and INSTALL_BLENDER and not BLENDER and STAGE in ('post_gpu', 'all'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', 'blender', 'libegl1', 'libgbm1'], check=True)
    BLENDER = shutil.which('blender') or ''
if EXECUTE and BACKEND == 'tripo':
    if not EXECUTE_PAID_API:
        raise RuntimeError('Tripo requires explicit EXECUTE_PAID_API=True')
    key = userdata.get('TRIPO_API_KEY')
    if not key:
        raise RuntimeError('Missing TRIPO_API_KEY in Colab Secrets')
    os.environ['TRIPO_API_KEY'] = key
check_cmd = [sys.executable, str(REPO / 'scripts/anime3d_preflight.py'), '--root', DRIVE_PROJECT,
             '--asset-id', ASSET_ID, '--spec', SPEC, '--backend', BACKEND, '--stage', STAGE,
             '--report', f'{DRIVE_PROJECT}/02_3d/colab/{ASSET_ID}/qa/preflight.json']
if BACKEND_CONFIG:
    check_cmd += ['--backend-config', BACKEND_CONFIG]
if RIG_BACKEND != 'none':
    check_cmd += ['--rig-backend', RIG_BACKEND]
if GENERATOR_EXECUTABLE:
    check_cmd += ['--generator-executable', GENERATOR_EXECUTABLE]
if BLENDER:
    check_cmd += ['--blender', BLENDER]
preflight = subprocess.run(check_cmd, check=False)
if EXECUTE and preflight.returncode:
    raise RuntimeError('Preflight blocked execution. Resolve the listed source/runtime checks first.')


## N02–N08：首次建模、外觀品質閘門、綁骨與 Blender 驗收

- `EXECUTE=False` 僅規劃；`EXECUTE=True` 才執行已安裝的開源模型。付費 Tripo 必須另外明確授權。
- `character-colab/2` 必須先提供已核准 `front`、`side+back`（或 `turnaround`）與獨立 `three_quarter_45`，並在 Drive 的個別 spec 設定 `reference_views`。
- `STAGE=pre_gpu`：CPU 來源／模型環境檢查，不要求正在使用 GPU 或已安裝 Blender。
- `STAGE=gpu`：先生成來源 GLB，外觀報告未齊即停在 `REVIEW_REQUIRED`，**不得進入綁骨**；GPU guard 仍須釋放資源。
- 另由真人或確實部署的視覺評估器輸出 `qa/appearance_review.json`，內容必須有模型與參考圖 SHA-256、五維分數、評估方法與評估者。缺角度才建議 MV-Adapter；幾何、材質、身份問題分別處理，最多兩次重新生成並使用新 asset_id。
- `STAGE=post_gpu`：品質門檻達標才允許 SkinTokens／Blender QA。**SkinTokens 本身可能需要 GPU**；若使用 `RIG_BACKEND=skintokens`，必須重新使用具相容 CUDA/VRAM 的執行環境，不能把整個 post_gpu 誤稱純 CPU。若已完成綁骨而只進 Blender，可用 CPU runtime。
- 模型、紋理、骨架、動作與 `.glb/.blend` 都需驗證，且最終仍要人工核准；個別檔案只寫 Drive。

GPU 斷線前必須檢查 Colab guard 的 `_gpu_runtime_status.json`，確認實際呼叫 unassign。此處規範不代表已實際執行 GPU、MV-Adapter 或 Texture Refinement。


In [ ]:
cmd = [sys.executable, str(REPO / 'scripts/anime3d_universal.py'), '--root', DRIVE_PROJECT,
       '--asset-id', ASSET_ID, '--spec', SPEC, '--backend', BACKEND, '--pipeline', PIPELINE]
_stage = globals().get('STAGE', 'all')
if _stage != 'all':
    cmd += ['--stage', _stage]
if EXECUTE:
    cmd.append('--execute')
if BACKEND == 'tripo' and EXECUTE_PAID_API:
    cmd.append('--allow-paid-tripo')
if BACKEND_CONFIG:
    cmd += ['--backend-config', BACKEND_CONFIG]
if RIG_BACKEND != 'none':
    cmd += ['--rig-backend', RIG_BACKEND]
if GENERATOR_EXECUTABLE:
    cmd += ['--generator-executable', GENERATOR_EXECUTABLE]
if BLENDER and _stage in ('post_gpu', 'all'):
    cmd += ['--blender', BLENDER]
_batch_queue = globals().get('BATCH_QUEUE', '')
_auto_shutdown = globals().get('AUTO_SHUTDOWN_GPU', False)
if _batch_queue:
    _guard_cmd = [sys.executable, str(REPO / 'scripts/anime3d_colab_guard.py'), '--root', DRIVE_PROJECT,
                  '--queue', _batch_queue, '--stage', _stage,
                  '--idle-timeout', str(globals().get('GPU_IDLE_TIMEOUT_SEC', 180))]
    if BACKEND_CONFIG:
        _guard_cmd += ['--backend-config', BACKEND_CONFIG]
    if BLENDER and _stage in ('post_gpu', 'all'):
        _guard_cmd += ['--blender', BLENDER]
    if not EXECUTE:
        _guard_cmd.append('--dry-run')
    if not _auto_shutdown:
        _guard_cmd.append('--no-auto-shutdown')
    subprocess.run(_guard_cmd, check=True)
elif EXECUTE and _stage == 'gpu' and _auto_shutdown:
    sys.path.insert(0, str(REPO / 'scripts'))
    from anime3d_colab_guard import GpuIdleWatchdog, shutdown_colab_runtime, write_runtime_status
    _drive_root = Path(DRIVE_PROJECT)
    write_runtime_status(_drive_root, active=True, reason='gpu_stage_started', unassign_invoked=False)
    _wd = GpuIdleWatchdog(idle_timeout_sec=globals().get('GPU_IDLE_TIMEOUT_SEC', 180),
                          shutdown_fn=lambda r: shutdown_colab_runtime(r, root=_drive_root))
    _wd.set_busy(True)
    _wd.start()
    try:
        subprocess.run(cmd, check=True)  # 失敗必須讓 Colab 顯示錯誤，不能只印 Exit code
    finally:
        _wd.stop()
        shutdown_colab_runtime('gpu_stage_finished', root=_drive_root)
else:
    subprocess.run(cmd, check=True)  # 失敗必須讓 Colab 顯示錯誤，不能只印 Exit code


In [ ]:
out = Path(DRIVE_PROJECT) / '02_3d/colab' / ASSET_ID
for name in ['typed_state.json', 'qa/reference_views.json', 'qa/appearance_review.json',
             'qa/appearance_gate.json', 'qa/typed_metrics.json', 'manifest.json']:
    target = out / name
    if target.is_file():
        result = json.loads(target.read_text(encoding='utf8'))
        keys = ['status', 'qa_status', 'automated_pass', 'passed', 'missing',
                'next_action', 'allow_rig', 'reference_sha256', 'model_sha256']
        print(name, json.dumps({k: result[k] for k in keys if k in result}, ensure_ascii=False))
print('逐項驗收臉部、服裝、配件、材質、輪廓、動畫與來源一致性；自動 QA 不代表人工核准。')
